# 2016 Local Government Election Data Cleaning

This notebook prepares the 2016 election results for analysis. It loads the raw CSV, standardizes column names, calculates voter turnout, aggregates records at municipality and party level, and saves a cleaned CSV file.

## Import pandas

`pandas` provides the DataFrame tools used throughout the cleaning process.

In [1]:
# Import pandas for tabular data cleaning and aggregation.
import pandas as pd
from pathlib import Path

# Paths are relative to the project folder, so this notebook runs on any computer.
# It works whether Jupyter is opened in the notebooks/ folder or in the project root.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
CLEAN_DIR = PROJECT_DIR / "data" / "cleaned"

## Load the raw data

Read the raw 2016 election results from `data/raw/` (zipped to stay under GitHub's file-size limit). Displaying `df` provides an initial view of the imported data.

In [2]:
# Load the raw 2016 election results (zipped to stay under GitHub's file-size limit).
df = pd.read_csv(RAW_DIR / "2016_LGE.csv.zip")

# Display the imported rows and columns for an initial data check.
df

,Province,Municipality,Ward,VotingDistrict,VotingStationName,RegisteredVoters,BallotType,SpoiltVotes,PartyName,TotalValidVotes,DateGenerated
0,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,AFRICAN CHRISTIAN DEMOCRATIC PARTY,3,2/16/2017 5:01:24 PM
1,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,AFRICAN INDEPENDENT CONGRESS,19,2/16/2017 5:01:24 PM
2,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,AFRICAN NATIONAL CONGRESS,347,2/16/2017 5:01:24 PM
3,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,CONGRESS OF THE PEOPLE,7,2/16/2017 5:01:24 PM
4,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,DEMOCRATIC ALLIANCE,751,2/16/2017 5:01:24 PM
...,...,...,...,...,...,...,...,...,...,...,...
653058,Western Cape,WC053 - Beaufort West,Ward 10503007,98180040,TWEERIVIERE FARM HOUSE,154,Ward,2,INDEPENDENT CIVIC ORGANISATION OF SOUTH AFRICA,0,2/16/2017 5:01:24 PM
653059,Western Cape,WC053 - Beaufort West,Ward 10503007,98180040,TWEERIVIERE FARM HOUSE,154,Ward,2,KAROO DEMOCRATIC FORCE,0,2/16/2017 5:01:24 PM
653060,Western Cape,WC053 - Beaufort West,Ward 10503007,98180040,TWEERIVIERE FARM HOUSE,154,Ward,2,PAN AFRICANIST CONGRESS OF AZANIA,0,2/16/2017 5:01:24 PM
653061,Western Cape,WC053 - Beaufort West,Ward 10503007,98180040,TWEERIVIERE FARM HOUSE,154,Ward,2,SOUTH AFRICAN RELIGIOUS CIVIC ORGANISATION,0,2/16/2017 5:01:24 PM


## Inspect the original columns

Review the column names before transforming the dataset. This helps identify verbose or inconsistent labels that should be standardized.

In [3]:
# Inspect the source column names before renaming any fields.
df.columns

Index(['Province', 'Municipality', 'Ward', 'VotingDistrict',
       'VotingStationName', 'RegisteredVoters', 'BallotType', 'SpoiltVotes',
       'PartyName', 'TotalValidVotes', 'DateGenerated'],
      dtype='str')

## Standardize column names

Rename the party and valid-vote columns to shorter, more readable labels. The values remain unchanged; only the column labels are updated.

In [4]:
# Use concise, analysis-friendly names for the party and vote columns.
df = df.rename(columns={'PartyName':'Party','TotalValidVotes':'Valid Votes Cast'})

## Calculate initial voter turnout

Calculate valid votes as a proportion of registered voters. This first calculation is an initial inspection and is later replaced with the municipality-level turnout measure after aggregation.

In [5]:
# Estimate turnout as a proportion for the initial row-level inspection.
df['% Voter Turnout'] = round(df['Valid Votes Cast']/df['RegisteredVoters'],2)

# Display the intermediate result before aggregation.
df

,Province,Municipality,Ward,VotingDistrict,VotingStationName,RegisteredVoters,BallotType,SpoiltVotes,Party,Valid Votes Cast,DateGenerated,% Voter Turnout
0,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,AFRICAN CHRISTIAN DEMOCRATIC PARTY,3,2/16/2017 5:01:24 PM,0.00
1,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,AFRICAN INDEPENDENT CONGRESS,19,2/16/2017 5:01:24 PM,0.01
2,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,AFRICAN NATIONAL CONGRESS,347,2/16/2017 5:01:24 PM,0.16
3,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,CONGRESS OF THE PEOPLE,7,2/16/2017 5:01:24 PM,0.00
4,Eastern Cape,BUF - Buffalo City,Ward 29200001,10590151,PEFFERVILLE CLINIC,2230,PR,21,DEMOCRATIC ALLIANCE,751,2/16/2017 5:01:24 PM,0.34
...,...,...,...,...,...,...,...,...,...,...,...,...
653058,Western Cape,WC053 - Beaufort West,Ward 10503007,98180040,TWEERIVIERE FARM HOUSE,154,Ward,2,INDEPENDENT CIVIC ORGANISATION OF SOUTH AFRICA,0,2/16/2017 5:01:24 PM,0.00
653059,Western Cape,WC053 - Beaufort West,Ward 10503007,98180040,TWEERIVIERE FARM HOUSE,154,Ward,2,KAROO DEMOCRATIC FORCE,0,2/16/2017 5:01:24 PM,0.00
653060,Western Cape,WC053 - Beaufort West,Ward 10503007,98180040,TWEERIVIERE FARM HOUSE,154,Ward,2,PAN AFRICANIST CONGRESS OF AZANIA,0,2/16/2017 5:01:24 PM,0.00
653061,Western Cape,WC053 - Beaufort West,Ward 10503007,98180040,TWEERIVIERE FARM HOUSE,154,Ward,2,SOUTH AFRICAN RELIGIOUS CIVIC ORGANISATION,0,2/16/2017 5:01:24 PM,0.00


## Aggregate the election records

Group records by province, municipality, party, and ballot type. Sum valid votes and registered voters so that each group has one consolidated record.


The registered-voter count is then replaced with the maximum value recorded for each municipality. This avoids repeatedly adding the same municipality-wide denominator across ward-level records.


`DC 40%` records are excluded because they are not part of the intended turnout calculation.

In [6]:
# Consolidate vote totals by geographic area, party, and ballot type.
df = (
    df
    .groupby(['Province', 'Municipality','Party','BallotType'], as_index=False)
    .agg(
        ValidVotesCast=('Valid Votes Cast', 'sum'),
        RegisteredVoters=('RegisteredVoters','sum') # Turnout from each ward
    )
)

# Keep one municipality-level denominator instead of summing repeated ward values.
df['RegisteredVoters'] = (
    df
    .groupby(['Province', 'Municipality'])['RegisteredVoters']
    .transform('max')
)

# Exclude DC 40% records from the turnout calculation.
df = df[df['BallotType']!="DC 40%"]

# Review the aggregated and filtered dataset.
df

,Province,Municipality,Party,BallotType,ValidVotesCast,RegisteredVoters
0,Eastern Cape,BUF - Buffalo City,AFRICAN CHRISTIAN DEMOCRATIC PARTY,PR,1244,419044
1,Eastern Cape,BUF - Buffalo City,AFRICAN CHRISTIAN DEMOCRATIC PARTY,Ward,1275,419044
2,Eastern Cape,BUF - Buffalo City,AFRICAN INDEPENDENT CONGRESS,PR,8869,419044
3,Eastern Cape,BUF - Buffalo City,AFRICAN INDEPENDENT CONGRESS,Ward,6731,419044
4,Eastern Cape,BUF - Buffalo City,AFRICAN NATIONAL CONGRESS,PR,136354,419044
...,...,...,...,...,...,...
5680,Western Cape,WC053 - Beaufort West,PAN AFRICANIST CONGRESS OF AZANIA,Ward,61,26027
5681,Western Cape,WC053 - Beaufort West,SOUTH AFRICAN RELIGIOUS CIVIC ORGANISATION,PR,47,26027
5682,Western Cape,WC053 - Beaufort West,SOUTH AFRICAN RELIGIOUS CIVIC ORGANISATION,Ward,31,26027
5684,Western Cape,WC053 - Beaufort West,VRYHEIDSFRONT PLUS,PR,114,26027


## Recalculate turnout as a percentage

Compute turnout after aggregation and express it as a percentage rounded to two decimal places. The denominator now represents the municipality-level registered-voter total.

In [7]:
# Convert the aggregated turnout ratio to a percentage.
df['% Voter Turnout'] = round((df['ValidVotesCast']/df['RegisteredVoters'])*100,2)

# Display the percentage turnout values for review.
df

,Province,Municipality,Party,BallotType,ValidVotesCast,RegisteredVoters,% Voter Turnout
0,Eastern Cape,BUF - Buffalo City,AFRICAN CHRISTIAN DEMOCRATIC PARTY,PR,1244,419044,0.30
1,Eastern Cape,BUF - Buffalo City,AFRICAN CHRISTIAN DEMOCRATIC PARTY,Ward,1275,419044,0.30
2,Eastern Cape,BUF - Buffalo City,AFRICAN INDEPENDENT CONGRESS,PR,8869,419044,2.12
3,Eastern Cape,BUF - Buffalo City,AFRICAN INDEPENDENT CONGRESS,Ward,6731,419044,1.61
4,Eastern Cape,BUF - Buffalo City,AFRICAN NATIONAL CONGRESS,PR,136354,419044,32.54
...,...,...,...,...,...,...,...
5680,Western Cape,WC053 - Beaufort West,PAN AFRICANIST CONGRESS OF AZANIA,Ward,61,26027,0.23
5681,Western Cape,WC053 - Beaufort West,SOUTH AFRICAN RELIGIOUS CIVIC ORGANISATION,PR,47,26027,0.18
5682,Western Cape,WC053 - Beaufort West,SOUTH AFRICAN RELIGIOUS CIVIC ORGANISATION,Ward,31,26027,0.12
5684,Western Cape,WC053 - Beaufort West,VRYHEIDSFRONT PLUS,PR,114,26027,0.44


## Check for duplicate rows

Inspect duplicate records without changing the DataFrame. The final `df` display makes it possible to compare the data before and after the duplicate check.

In [8]:
# Inspect duplicate rows without assigning the result back to df.
df.drop_duplicates()

# Display the current DataFrame after the duplicate check.
df

,Province,Municipality,Party,BallotType,ValidVotesCast,RegisteredVoters,% Voter Turnout
0,Eastern Cape,BUF - Buffalo City,AFRICAN CHRISTIAN DEMOCRATIC PARTY,PR,1244,419044,0.30
1,Eastern Cape,BUF - Buffalo City,AFRICAN CHRISTIAN DEMOCRATIC PARTY,Ward,1275,419044,0.30
2,Eastern Cape,BUF - Buffalo City,AFRICAN INDEPENDENT CONGRESS,PR,8869,419044,2.12
3,Eastern Cape,BUF - Buffalo City,AFRICAN INDEPENDENT CONGRESS,Ward,6731,419044,1.61
4,Eastern Cape,BUF - Buffalo City,AFRICAN NATIONAL CONGRESS,PR,136354,419044,32.54
...,...,...,...,...,...,...,...
5680,Western Cape,WC053 - Beaufort West,PAN AFRICANIST CONGRESS OF AZANIA,Ward,61,26027,0.23
5681,Western Cape,WC053 - Beaufort West,SOUTH AFRICAN RELIGIOUS CIVIC ORGANISATION,PR,47,26027,0.18
5682,Western Cape,WC053 - Beaufort West,SOUTH AFRICAN RELIGIOUS CIVIC ORGANISATION,Ward,31,26027,0.12
5684,Western Cape,WC053 - Beaufort West,VRYHEIDSFRONT PLUS,PR,114,26027,0.44


## Find turnout for each municipality

For each municipality, keep the highest turnout value observed across its grouped party and ballot records. This creates a lookup table containing one turnout value per municipality.

In [9]:
# Select one municipality turnout value for use in the final dataset.
municipality_turnout = (
    df
    .groupby(['Municipality'], as_index=False)['% Voter Turnout']
    .max()) # Turnout from each municipality

# Review the one-row-per-municipality lookup table.
municipality_turnout

,Municipality,% Voter Turnout
0,BUF - Buffalo City,32.54
1,CPT - City of Cape Town,42.10
2,EC101 - Camdeboo,30.83
3,EC102 - Blue Crane Route,36.85
4,EC104 - Makana,32.50
...,...,...
208,WC047 - Bitou,32.80
209,WC048 - Knysna,31.21
210,WC051 - Laingsburg,32.44
211,WC052 - Prince Albert,25.02


> **Note added after the EDA: this turnout column should not be used as turnout.**
>
> The calculation above divides **one party's** valid votes by registered voters, and then keeps the **highest** value per municipality. The result is the *leading party's votes as a share of registered voters*, not turnout (all votes cast ÷ registered voters). For Johannesburg it gives 25.18%, while the true 2016 turnout from the same raw file is 56.87%.
>
> The EDA notebook detects this with an independent check (section 3) and excludes 2016 turnout from all turnout analysis. 2016 **vote counts** are correct and are used throughout. We have left this notebook's logic unchanged so that the cleaned file used in the analysis can be reproduced exactly; a correct turnout would be (sum of valid votes across all parties + spoilt votes) ÷ registered voters on the PR ballot.

## Attach municipality turnout to every record

Remove the temporary turnout column and merge the municipality-level turnout table back into the detailed election records.


The registered-voter denominator is removed from the final dataset because the municipality turnout percentage is now the retained summary measure.

In [10]:
# Replace the detailed turnout field with the municipality-level lookup value.
df = df.drop(columns=['% Voter Turnout']).merge(
    municipality_turnout,
    on='Municipality',
    how='left'
 )

# The final file keeps the turnout percentage rather than its denominator.
df = df.drop(columns=['RegisteredVoters'])

# Display the cleaned records before export.
df

,Province,Municipality,Party,BallotType,ValidVotesCast,% Voter Turnout
0,Eastern Cape,BUF - Buffalo City,AFRICAN CHRISTIAN DEMOCRATIC PARTY,PR,1244,32.54
1,Eastern Cape,BUF - Buffalo City,AFRICAN CHRISTIAN DEMOCRATIC PARTY,Ward,1275,32.54
2,Eastern Cape,BUF - Buffalo City,AFRICAN INDEPENDENT CONGRESS,PR,8869,32.54
3,Eastern Cape,BUF - Buffalo City,AFRICAN INDEPENDENT CONGRESS,Ward,6731,32.54
4,Eastern Cape,BUF - Buffalo City,AFRICAN NATIONAL CONGRESS,PR,136354,32.54
...,...,...,...,...,...,...
3730,Western Cape,WC053 - Beaufort West,PAN AFRICANIST CONGRESS OF AZANIA,Ward,61,29.58
3731,Western Cape,WC053 - Beaufort West,SOUTH AFRICAN RELIGIOUS CIVIC ORGANISATION,PR,47,29.58
3732,Western Cape,WC053 - Beaufort West,SOUTH AFRICAN RELIGIOUS CIVIC ORGANISATION,Ward,31,29.58
3733,Western Cape,WC053 - Beaufort West,VRYHEIDSFRONT PLUS,PR,114,29.58


## Export the cleaned dataset

Save the finished DataFrame as `2016_LGE_Cleaned.csv` so it can be used by later analysis or visualization steps.

In [11]:
# Export the cleaned data for downstream analysis.
df.to_csv(CLEAN_DIR / '2016_LGE_Cleaned.csv')